# Assignment 2, notebook 3: Models, validation, ablation and feature selection (Part C)

**Setup:** train on FY2014-17, tune on FY2018, test once on FY2019 (time-based split from notebook 2).

**Steps**
1. Compare four models
2. Check the validation strategy (time-based vs random split)
3. Tune hyperparameters
4. Ablation studies: feature groups, brand-new lenders, single features
5. Feature selection
6. Final test on FY2019: ranking metrics, precision / recall / F1, confusion matrix, risk deciles
7. Save the model and results for notebook 4

Experiments that did not make the final model (resampling, outside data, other targets) are in `03b_experiments.ipynb`.

In [1]:
import json, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss, roc_curve,
                             precision_recall_curve, confusion_matrix, classification_report)
from sklearn.model_selection import GroupKFold, train_test_split
import lightgbm as lgb
warnings.filterwarnings("ignore")

In [2]:
TEAL, GREY, RED = "#0E7C86", "#B8BFCC", "#C0392B"
PROC = Path("../../data/processed"); OUT = Path("../outputs")
tr, va, te = (pd.read_parquet(PROC / f"a2_{s}.parquet") for s in ["train", "valid", "test"])
GROUPS = json.load(open(OUT / "a2_feature_groups.json"))
FEATS = sum(GROUPS.values(), [])
ytr, yva, yte = tr["bad"], va["bad"], te["bad"]
print(len(FEATS), "features | train", tr.shape, "| valid", va.shape, "| test", te.shape)
print(f"Bad-loan rate: train {ytr.mean():.2%}, valid {yva.mean():.2%}, test {yte.mean():.2%}")

101 features | train (214087, 107) | valid (54172, 107) | test (45657, 107)
Bad-loan rate: train 7.74%, valid 8.14%, test 6.59%


### Helper functions
*Why:* one metrics function is used everywhere so every model is judged the same way.
- **AUC** (0.5 = coin flip, 1 = perfect) and **PR-AUC** (a random model scores the bad-loan rate) measure how well loans are ranked.
- **KS** is the largest gap between bad and good loans along the ranking (a common credit metric). **Brier** measures how close the predicted probabilities are to what happened (lower is better).
- **Bad caught in the riskiest 10% / 20%** and **lift** answer the business question: if the team can only review a fixed share of loans, how many bad ones do they find?

In [3]:
# helpers

def metrics(y, p):
    fpr, tpr, _ = roc_curve(y, p); order = np.argsort(-p); n = len(y)
    r = {"AUC": roc_auc_score(y, p), "PR-AUC": average_precision_score(y, p),
         "KS": (tpr - fpr).max(), "Brier": brier_score_loss(y, p)}
    for k in (10, 20):
        r[f"bad caught in top {k}%"] = y.values[order[: int(n * k / 100)]].sum() / y.sum()
    r["lift top 10%"] = r["bad caught in top 10%"] / 0.10
    return r

BASE = dict(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=100, subsample=0.8,
            subsample_freq=1, colsample_bytree=0.8, reg_lambda=5, random_state=42, n_jobs=-1, verbose=-1)
def lgbm(**kw): return lgb.LGBMClassifier(**{**BASE, **kw})

## 1. Compare four models on FY2018
*Why:* logistic regression is the simple, explainable benchmark; a random forest is a robust middle option; gradient boosting (LightGBM) is usually strongest on tabular data. A class-weighted LightGBM answers the imbalance question.

In [4]:
res = {}
lr = make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=2000, class_weight="balanced")).fit(tr[FEATS], ytr)
res["Logistic regression"] = metrics(yva, lr.predict_proba(va[FEATS])[:, 1])

rf = RandomForestClassifier(n_estimators=200, max_depth=14, min_samples_leaf=50, max_features="sqrt",
                            class_weight="balanced_subsample", n_jobs=-1, random_state=42).fit(tr[FEATS], ytr)
res["Random forest"] = metrics(yva, rf.predict_proba(va[FEATS])[:, 1])

gb = lgbm().fit(tr[FEATS], ytr)
res["LightGBM"] = metrics(yva, gb.predict_proba(va[FEATS])[:, 1])

pos_w = (1 - ytr.mean()) / ytr.mean()
gbw = lgbm(scale_pos_weight=pos_w).fit(tr[FEATS], ytr)
res["LightGBM (class-weighted)"] = metrics(yva, gbw.predict_proba(va[FEATS])[:, 1])

cmp = pd.DataFrame(res).T
display(cmp.round(4))
px.bar(cmp[["AUC", "PR-AUC"]].reset_index().melt("index"), x="index", y="value", color="variable", barmode="group",
       text_auto=".3f", color_discrete_sequence=[TEAL, GREY], template="plotly_white",
       title="Model comparison on FY2018 (validation)", labels={"index": "", "value": ""}).show()

,AUC,PR-AUC,KS,Brier,bad caught in top 10%,bad caught in top 20%,lift top 10%
Logistic regression,0.6522,0.1883,0.2059,0.4101,0.2507,0.3530,2.5074
Random forest,0.7559,0.2354,0.3781,0.2143,0.3158,0.4935,3.1580
LightGBM,0.7621,0.2577,0.3839,0.0680,0.3299,0.4988,3.2986
LightGBM (class-weighted),0.7602,0.2527,0.3866,0.2181,0.3262,0.4988,3.2623


**Reading:** LightGBM ranks best. The class-weighted version ranks about as well but its probabilities are inflated (much worse Brier), so we keep the unweighted model and handle the cut-off separately.

## 2. Validation strategy: does a random split flatter the model?
*Why:* in real use the model scores future loans, so our main test is time-based. This shows what a random 80/20 split would have reported.

In [5]:
pool = pd.concat([tr, va])
Xa, Xb, ya, yb = train_test_split(pool[FEATS], pool["bad"], test_size=0.2, random_state=0, stratify=pool["bad"])
m = lgbm().fit(Xa, ya)
print("Random 80/20 split AUC:", round(roc_auc_score(yb, m.predict_proba(Xb)[:, 1]), 4),
      "| time-based (train FY14-17 -> FY2018):", round(res["LightGBM"]["AUC"], 4))

Random 80/20 split AUC: 0.7515 | time-based (train FY14-17 -> FY2018): 0.7621


## 3. Hyperparameter tuning
*Why:* try 8 random settings; training stops early when the FY2018 score stops improving, which also picks the number of trees. Trials are ranked on PR-AUC, which suits the imbalanced target.

In [6]:
rng = np.random.default_rng(7)
space = dict(num_leaves=[15, 31, 63], learning_rate=[0.03, 0.05, 0.1], min_child_samples=[50, 100, 300],
             subsample=[0.7, 0.85, 1.0], colsample_bytree=[0.6, 0.8, 1.0], reg_lambda=[0, 5, 20])
trials = []
for i in range(8):
    p = {k: rng.choice(v).item() for k, v in space.items()}
    m = lgbm(n_estimators=1500, **p)
    m.fit(tr[FEATS], ytr, eval_set=[(va[FEATS], yva)], eval_metric="average_precision",
          callbacks=[lgb.early_stopping(50, verbose=False)])
    pv = m.predict_proba(va[FEATS])[:, 1]
    trials.append({**p, "n_trees": m.best_iteration_, "valid_AUC": roc_auc_score(yva, pv),
                   "valid_PR_AUC": average_precision_score(yva, pv)})
tuning = pd.DataFrame(trials).sort_values("valid_PR_AUC", ascending=False)
display(tuning.round(4))

best = {k: tuning.iloc[0][k] for k in space}
best = {k: (int(v) if k in ("num_leaves", "min_child_samples") else float(v)) for k, v in best.items()}
NT = int(tuning.iloc[0]["n_trees"])
tuned = lambda **kw: lgbm(n_estimators=NT, **{**best, **kw})

full = tuned().fit(tr[FEATS], ytr)
pfull = full.predict_proba(va[FEATS])[:, 1]; mfull = metrics(yva, pfull)
print("Best settings:", best, "| trees:", NT, "| valid AUC:", round(mfull["AUC"], 4), "| valid PR-AUC:", round(mfull["PR-AUC"], 4))

,num_leaves,learning_rate,min_child_samples,subsample,colsample_bytree,reg_lambda,n_trees,valid_AUC,valid_PR_AUC
7,63,0.03,300,0.85,0.6,0,329,0.7628,0.2602
1,63,0.03,50,0.70,0.6,20,504,0.7621,0.2573
6,63,0.05,100,1.00,0.8,0,282,0.7612,0.2568
3,15,0.05,300,0.70,0.8,0,513,0.7607,0.2566
2,63,0.03,100,1.00,0.6,20,245,0.7613,0.2559
4,63,0.03,300,0.85,0.8,5,234,0.7606,0.2554
0,63,0.05,300,1.00,0.8,20,236,0.7600,0.2541
5,31,0.05,100,1.00,1.0,20,333,0.7589,0.2521


Best settings: {'num_leaves': 63, 'learning_rate': 0.03, 'min_child_samples': 300, 'subsample': 0.85, 'colsample_bytree': 0.6, 'reg_lambda': 0.0} | trees: 329 | valid AUC: 0.7628 | valid PR-AUC: 0.2602


## 4. Ablation study A: which feature groups matter?
*Why:* remove one group at a time (how much does accuracy fall?) and use each group alone (how much does it carry by itself?).

In [7]:
rows = [{"scenario": "All features", "AUC": mfull["AUC"], "PR-AUC": mfull["PR-AUC"], "features": len(FEATS)}]
for g, fs in GROUPS.items():
    keep = [f for f in FEATS if f not in fs]
    p = tuned().fit(tr[keep], ytr).predict_proba(va[keep])[:, 1]
    rows.append({"scenario": f"Without {g}", "AUC": roc_auc_score(yva, p), "PR-AUC": average_precision_score(yva, p), "features": len(keep)})
for g, fs in GROUPS.items():
    p = tuned().fit(tr[fs], ytr).predict_proba(va[fs])[:, 1]
    rows.append({"scenario": f"Only {g}", "AUC": roc_auc_score(yva, p), "PR-AUC": average_precision_score(yva, p), "features": len(fs)})
abl = pd.DataFrame(rows)
abl["AUC change vs all"] = abl["AUC"] - mfull["AUC"]
display(abl.round(4))
px.bar(abl.iloc[1:].sort_values("AUC change vs all"), x="AUC change vs all", y="scenario", orientation="h",
       color_discrete_sequence=[TEAL], template="plotly_white", title="Ablation by feature group: change in AUC (FY2018)").show()

,scenario,AUC,PR-AUC,features,AUC change vs all
0,All features,0.7628,0.2602,101,0.0000
1,Without loan,0.7468,0.2452,94,-0.0160
2,Without borrower,0.7600,0.2568,85,-0.0028
3,Without industry_geo,0.7512,0.2443,35,-0.0116
4,Without product,0.7617,0.2577,93,-0.0011
5,Without lender,0.7488,0.2043,97,-0.0139
6,Only loan,0.7227,0.1754,7,-0.0401
7,Only borrower,0.6335,0.1246,16,-0.1292
8,Only industry_geo,0.6547,0.1373,66,-0.1081
9,Only product,0.5501,0.0921,8,-0.2127


## 5. Ablation study B: brand-new lenders
*Why:* lender history helps for lenders we have seen, but a new lender has none. We train on some lenders and score the others as if they were new (their lender-history features reset to what a newcomer would have).

In [8]:
LEND = GROUPS["lender"]; prior = ytr.mean(); out = []
nol = [f for f in FEATS if f not in LEND]
for a, b in GroupKFold(5).split(tr, ytr, groups=tr["bankname"]):
    A, B = tr.iloc[a], tr.iloc[b].copy()
    B["lender_te"] = prior; B["lender_log_volume"] = 0.0; B["lender_seen"] = 0
    pw = tuned().fit(A[FEATS], A["bad"]).predict_proba(B[FEATS])[:, 1]
    pn = tuned().fit(A[nol], A["bad"]).predict_proba(B[nol])[:, 1]
    out.append((roc_auc_score(B["bad"], pw), roc_auc_score(B["bad"], pn)))
newlender = pd.DataFrame(out, columns=["with lender features", "without lender features"])
display(newlender.round(4)); print("Average:", newlender.mean().round(4).to_dict())

,with lender features,without lender features
0,0.6950,0.7205
1,0.6324,0.6434
2,0.6743,0.7001
3,0.6845,0.6780
4,0.7137,0.7404


Average: {'with lender features': 0.68, 'without lender features': 0.6965}


## 6. Feature selection
*Why:* many of the 101 features are small state or sector flags. We rank features by **permutation importance** (shuffle one feature, measure how much validation AUC falls), refit with only the top *k*, and keep the smallest *k* within 0.002 AUC of the best.

In [9]:
rng = np.random.default_rng(1); base_auc = roc_auc_score(yva, pfull); Xv = va[FEATS].copy(); rows = []
for f in FEATS:
    drops = []
    for _ in range(2):
        orig = Xv[f].values.copy(); Xv[f] = rng.permutation(orig)
        drops.append(base_auc - roc_auc_score(yva, full.predict_proba(Xv)[:, 1])); Xv[f] = orig
    rows.append((f, np.mean(drops)))
perm = pd.DataFrame(rows, columns=["feature", "AUC_drop"]).sort_values("AUC_drop", ascending=False).reset_index(drop=True)

ks = [5, 8, 12, 16, 20, 30, 45, 70, len(FEATS)]; krows = []
for k in ks:
    fs = perm["feature"].head(k).tolist()
    p = tuned().fit(tr[fs], ytr).predict_proba(va[fs])[:, 1]
    krows.append({"features": k, "AUC": roc_auc_score(yva, p), "PR-AUC": average_precision_score(yva, p)})
kt = pd.DataFrame(krows)
K = int(kt.loc[kt["AUC"] >= kt["AUC"].max() - 0.002, "features"].min())
SEL = perm["feature"].head(K).tolist()
print("Chosen number of features:", K); display(kt.round(4))

fig = px.line(kt, x="features", y="AUC", markers=True, log_x=True, template="plotly_white", title="Validation AUC by number of features kept")
fig.add_vline(x=K, line_dash="dot", line_color=RED); fig.show()
px.bar(perm.head(20).iloc[::-1], x="AUC_drop", y="feature", orientation="h", color_discrete_sequence=[TEAL],
       template="plotly_white", title="Permutation importance: AUC lost when the feature is shuffled (top 20)").show()

Chosen number of features: 45


,features,AUC,PR-AUC
0,5,0.7517,0.2531
1,8,0.7529,0.2528
2,12,0.7577,0.2545
3,16,0.7595,0.2568
4,20,0.7595,0.2568
5,30,0.7603,0.2569
6,45,0.7619,0.2578
7,70,0.7626,0.2586
8,101,0.7620,0.2573


## 7. Ablation study C: drop one top feature at a time
*Why:* checks that no single feature carries the model.

In [10]:
base_k = kt.loc[kt.features == K, "AUC"].iloc[0]; lo = []
for f in perm["feature"].head(8):
    fs = [x for x in SEL if x != f]
    p = tuned().fit(tr[fs], ytr).predict_proba(va[fs])[:, 1]
    lo.append({"dropped": f, "AUC": roc_auc_score(yva, p), "change": roc_auc_score(yva, p) - base_k})
pd.DataFrame(lo).round(4)

,dropped,AUC,change
0,lender_te,0.7588,-0.0031
1,rate_vs_year,0.7505,-0.0113
2,naics3_te,0.7598,-0.0021
3,log_amount,0.7607,-0.0011
4,lender_log_volume,0.7616,-0.0003
5,variable_rate,0.7629,0.0010
6,amount_per_job_log,0.7626,0.0007
7,revolving,0.7611,-0.0008


## 8. Final test on FY2019
*Why:* FY2019 has not been used until now, so this is the one honest estimate of performance on new loans.

In [11]:
final = tuned().fit(tr[SEL], ytr); pte = final.predict_proba(te[SEL])[:, 1]
tab = {"Logistic regression": metrics(yte, lr.predict_proba(te[FEATS])[:, 1]),
       "Random forest": metrics(yte, rf.predict_proba(te[FEATS])[:, 1]),
       "LightGBM, all features": metrics(yte, full.predict_proba(te[FEATS])[:, 1]),
       f"LightGBM, selected ({K}) - chosen": metrics(yte, pte)}
test_table = pd.DataFrame(tab).T
display(test_table.round(4))
print(f"Random-pick baseline for PR-AUC = bad-loan rate = {yte.mean():.3f}")

fpr, tpr, _ = roc_curve(yte, pte)
fig = go.Figure(go.Scatter(x=fpr, y=tpr, line=dict(color=TEAL, width=3), name="Selected model"))
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(dash="dot", color=GREY))
fig.update_layout(template="plotly_white", title=f"ROC curve, FY2019 test (AUC {roc_auc_score(yte, pte):.3f})",
                  xaxis_title="Share of good loans wrongly flagged", yaxis_title="Share of bad loans caught"); fig.show()

,AUC,PR-AUC,KS,Brier,bad caught in top 10%,bad caught in top 20%,lift top 10%
Logistic regression,0.6433,0.1432,0.1872,0.4681,0.2514,0.3721,2.5141
Random forest,0.7503,0.1851,0.3657,0.2156,0.3056,0.5071,3.0562
"LightGBM, all features",0.7527,0.1941,0.3737,0.0584,0.3216,0.5098,3.2158
"LightGBM, selected (45) - chosen",0.7528,0.1934,0.3749,0.0586,0.3169,0.5085,3.1693


Random-pick baseline for PR-AUC = bad-loan rate = 0.066


## 9. Precision, recall, F1 and the confusion matrix
The ranking metrics say how well loans are ordered. A lending team must also decide **where to draw the line**.
- **Precision:** of the loans we flag, how many really are bad. **Recall:** of all bad loans, how many we flag. **F1** balances the two.
- We compare a probability cut-off chosen on FY2018 (best F1, never chosen on the test year) with fixed review capacities (riskiest 5%, 10%, 20%, 30%).

In [12]:
pva = final.predict_proba(va[SEL])[:, 1]
prec, rec, thr = precision_recall_curve(yva, pva)
f1s = 2 * prec * rec / (prec + rec + 1e-12)
t_f1 = thr[np.argmax(f1s[:-1])]

def operating_point(name, flag):
    tn, fp, fn, tp = confusion_matrix(yte.values, flag).ravel()
    return {"policy": name, "loans flagged %": flag.mean() * 100, "precision": tp / (tp + fp), "recall": tp / (tp + fn),
            "F1": 2 * tp / (2 * tp + fp + fn), "TP": tp, "FP": fp, "FN": fn, "TN": tn}

ops = [operating_point(f"Probability cut-off {t_f1:.2f} (best F1 on FY2018)", pte >= t_f1)]
order = np.argsort(-pte)
for share in (0.05, 0.10, 0.20, 0.30):
    flag = np.zeros(len(pte), dtype=bool); flag[order[: int(len(pte) * share)]] = True
    ops.append(operating_point(f"Flag the riskiest {share:.0%}", flag))
op = pd.DataFrame(ops).set_index("policy")
display(op.round(3))
print(f"Base rate in FY2019: {yte.mean():.1%} (the precision of a random pick)")

flag20 = np.zeros(len(pte), dtype=bool); flag20[order[: int(len(pte) * 0.20)]] = True
cm = confusion_matrix(yte, flag20)
px.imshow(cm, text_auto=",", x=["Not flagged", "Flagged"], y=["Good loan", "Bad loan"], color_continuous_scale="Teal",
          title="Confusion matrix, FY2019: flag the riskiest 20% of loans", template="plotly_white").show()
print(classification_report(yte, flag20, target_names=["good", "bad"], digits=3))

p_, r_, _ = precision_recall_curve(yte, pte)
fig = go.Figure(go.Scatter(x=r_, y=p_, line=dict(color=TEAL, width=3), name="Model"))
fig.add_hline(y=yte.mean(), line_dash="dot", line_color=RED, annotation_text=f"Random pick: {yte.mean():.1%}")
fig.update_layout(template="plotly_white", title="Precision-recall curve, FY2019 test",
                  xaxis_title="Recall (share of bad loans caught)", yaxis_title="Precision (share of flagged loans that are bad)"); fig.show()

,loans flagged %,precision,recall,F1,TP,FP,FN,TN
policy,,,,,,,,
Probability cut-off 0.17 (best F1 on FY2018),13.520,0.195,0.400,0.262,1202,4971,1805,37679
Flag the riskiest 5%,4.998,0.246,0.187,0.212,561,1721,2446,40929
Flag the riskiest 10%,9.998,0.209,0.317,0.252,953,3612,2054,39038
Flag the riskiest 20%,19.999,0.167,0.508,0.252,1529,7602,1478,35048
Flag the riskiest 30%,30.000,0.142,0.647,0.233,1945,11752,1062,30898


Base rate in FY2019: 6.6% (the precision of a random pick)


              precision    recall  f1-score   support

        good      0.960     0.822     0.885     42650
         bad      0.167     0.508     0.252      3007

    accuracy                          0.801     45657
   macro avg      0.563     0.665     0.569     45657
weighted avg      0.907     0.801     0.844     45657



## 10. Risk deciles and calibration
*Why:* sort FY2019 loans into ten equal groups by predicted risk and compare predicted vs actual bad rates and dollars lost.

In [13]:
te2 = te.assign(p=pte)
te2["decile"] = pd.qcut(te2["p"].rank(method="first"), 10, labels=range(1, 11)).astype(int)
dec = te2.groupby("decile").agg(loans=("bad", "size"), predicted=("p", "mean"), actual=("bad", "mean"),
                                lost_usd_m=("grosschargeoffamount", lambda s: s.fillna(0).sum() / 1e6)).sort_index(ascending=False)
dec["share of bad loans"] = te2.groupby("decile")["bad"].sum().sort_index(ascending=False) / te2["bad"].sum()
dec["share of $ lost"] = dec["lost_usd_m"] / dec["lost_usd_m"].sum()
display(dec.round(3))

d_ = dec.reset_index()
fig = go.Figure([go.Bar(x=d_["decile"], y=d_["actual"] * 100, name="Actual bad rate (%)", marker_color=TEAL),
                 go.Scatter(x=d_["decile"], y=d_["predicted"] * 100, name="Predicted (%)", mode="lines+markers", line=dict(color=RED))])
fig.update_layout(template="plotly_white", title="Risk deciles, FY2019: predicted vs actual (decile 10 = riskiest)",
                  xaxis_title="Risk decile", yaxis_title="%"); fig.show()
print("Same scores against the 60-month target, AUC:", round(roc_auc_score(te["bad_60m"], pte), 4))

,loans,predicted,actual,lost_usd_m,share of bad loans,share of $ lost
decile,,,,,,
10,4566,0.290,0.209,96.163,0.317,0.227
9,4566,0.161,0.126,56.609,0.192,0.134
8,4565,0.120,0.091,47.524,0.138,0.112
7,4566,0.094,0.065,45.638,0.099,0.108
6,4565,0.075,0.050,41.593,0.076,0.098
5,4566,0.059,0.042,38.724,0.064,0.091
4,4566,0.045,0.032,30.311,0.049,0.072
3,4565,0.032,0.022,23.347,0.033,0.055
2,4566,0.021,0.015,23.416,0.023,0.055


Same scores against the 60-month target, AUC: 0.7628


## 11. Does it work across loan sizes?

In [14]:
te2["size_band"] = pd.cut(te2["grossapproval"], [0, 50e3, 150e3, 350e3, 1e6, 5e6],
                          labels=["Up to $50k", "$50-150k", "$150-350k", "$350k-1m", "Over $1m"])
te2.groupby("size_band", observed=True).apply(
    lambda g: pd.Series({"loans": len(g), "bad_rate": g.bad.mean(), "AUC": roc_auc_score(g.bad, g.p)})).round(3)

,loans,bad_rate,AUC
size_band,,,
Up to $50k,12522.0,0.094,0.709
$50-150k,10875.0,0.088,0.738
$150-350k,9438.0,0.063,0.711
$350k-1m,7316.0,0.028,0.735
Over $1m,5506.0,0.015,0.714


## 12. Save for notebook 4

In [16]:
# final.booster_.save_model(str(OUT / "a2_final_model.txt"))
# json.dump({"selected_features": SEL, "params": best, "n_trees": NT}, open(OUT / "a2_final_model_config.json", "w"), indent=1)
# te2[["approvalfy", "bankname", "bad", "bad_60m", "grossapproval", "grosschargeoffamount", "p"]].to_parquet(PROC / "a2_test_scores.parquet", index=False)
# test_table.to_csv(OUT / "a2_test_metrics.csv"); op.to_csv(OUT / "a2_operating_points.csv")
# abl.to_csv(OUT / "a2_ablation_groups.csv", index=False); perm.to_csv(OUT / "a2_permutation_importance.csv", index=False)
# kt.to_csv(OUT / "a2_topk_curve.csv", index=False); dec.to_csv(OUT / "a2_test_deciles.csv")
# print("saved")

## Summary

**Result on the test year (FY2019, never used before):** the chosen model, LightGBM with 45 selected features, scores **AUC 0.753** and **PR-AUC 0.193** (random pick: 0.066, so about 2.9 times better).

**What that means in practice**
- **Riskiest 10% of loans:** 32% of all bad loans (3.2 times a random pick) and 23% of the dollars lost.
- **Riskiest 20%:** about half of all bad loans (51%), with a precision of 17% against 6.6% for a random pick.
- **Use:** good enough to prioritise reviews, pricing and monitoring; not good enough to approve or decline loans on its own.

**How we got there**
- **Model comparison (FY2018):** LightGBM 0.762 > random forest 0.756 > logistic regression 0.652. Class weighting ranked no better and inflated the probabilities (Brier 0.218 vs 0.068), so the unweighted model is used.
- **Validation:** time-based (train FY2014-17, tune FY2018, test FY2019). A random split gave a similar score (0.752), so the time split is not hiding a problem.
- **Tuning:** 8 settings ranked on PR-AUC, all within 0.759-0.763 AUC; the model is not sensitive to them.
- **Ablation:** loan, lender and industry/place features matter most (removing each costs 0.012-0.016 AUC); borrower and product features add little once those are present. No single feature carries the model; the rate relative to that year's market is the largest single contributor.
- **New lenders:** on lenders with no history, AUC falls to about 0.68-0.70 and the lender features stop helping. The model is strongest for known lenders (99% of FY2018-19 loans).
- **Feature selection:** 45 features match the full 101 (test AUC 0.753 for both), so the simpler model is chosen.
- **Stability:** AUC 0.71-0.74 in every loan-size band; 0.763 against the 60-month target, so the target choice does not drive the result.

**Limits to carry forward**
- Predicted probabilities run high for FY2019 (riskiest decile: 29% predicted vs 21% actual), because those loans are younger. Notebook 4 recalibrates them.
- The ceiling is set by the data: public SBA data has no borrower financials. Notebook 3b tests resampling, other targets and outside data; none moves the result meaningfully.
- The loan-term field is excluded because it is rewritten after loans go bad (proof in notebook 3b, section 4a).